# Iceberg Schema Evolution Scenarios

This notebook demonstrates Apache Iceberg schema evolution patterns on AWS Glue:
1. Creating a base table
2. Adding columns
3. Dropping columns
4. Inserting old-schema data into the evolved table
5. Using branching for safe schema evolution
6. Loading old data that contains dropped columns

## Session Setup

Configure the Glue interactive session with Iceberg support.

In [9]:
%stop_session
# Update these values for your environment
%iam_role arn:aws:iam::038676220235:role/AWSGlueServiceNotebookRoleDefault

%glue_version 5.0
%worker_type G.1X
%number_of_workers 2
%idle_timeout 300
%region us-west-2
%%configure
{
  "--datalake-formats": "iceberg",
  "--conf": "spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions --conf spark.sql.catalog.glue_catalog=org.apache.iceberg.spark.SparkCatalog --conf spark.sql.catalog.glue_catalog.warehouse=s3://<YOUR_BUCKET>/warehouse/ --conf spark.sql.catalog.glue_catalog.type=glue --conf spark.sql.defaultCatalog=glue_catalog"
}

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
There is no current session.
Current iam_role is None
iam_role has been set to arn:aws:iam::038676220235:role/AWSGlueServiceNotebookRoleDefault.
Setting Glue version to: 5.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 2
Current idle_timeout is None minutes.
idle_timeout has been set to 300 minutes.
Previous region: us-west-2
Setting new region to: us-west-2
Region is set to: us-west-2
The following configurations have been updated: {'--datalake-formats': 'iceberg', '--conf': 'spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions --conf spark.sql.catalog.glue_c

In [1]:
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, col, current_timestamp
from pyspark.sql.types import StructType, StructField, LongType, StringType, TimestampType, IntegerType
from awsglue.context import GlueContext
from awsglue.job import Job

spark = SparkSession.builder.getOrCreate()
glueContext = GlueContext(spark.sparkContext)
job = Job(glueContext)

Trying to create a Glue session for the kernel.
Session Type: etl
Worker Type: G.1X
Number of Workers: 2
Idle Timeout: 300
Session ID: 642f67d0-d32a-4eac-8800-46679da3056a
Applying the following default arguments:
--glue_kernel_version 1.0.9
--enable-glue-datacatalog true
--datalake-formats iceberg
--conf spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions --conf spark.sql.catalog.glue_catalog=org.apache.iceberg.spark.SparkCatalog --conf spark.sql.catalog.glue_catalog.warehouse=s3://<YOUR_BUCKET>/warehouse/ --conf spark.sql.catalog.glue_catalog.type=glue --conf spark.sql.defaultCatalog=glue_catalog
Waiting for session 642f67d0-d32a-4eac-8800-46679da3056a to get into ready status...
Session 642f67d0-d32a-4eac-8800-46679da3056a has been created.



## Configuration

Set your database and table names here. Adjust the bucket/warehouse path in the session config above.

In [5]:
DATABASE = "jpmc_demo_warehouse"
TABLE_NAME = "events"
FULL_TABLE = f"glue_catalog.{DATABASE}.{TABLE_NAME}"

# Create the database if it doesn't exist
#spark.sql(f"CREATE DATABASE IF NOT EXISTS glue_catalog.{DATABASE}")
print(f"Using database: {DATABASE}")

Using database: jpmc_demo_warehouse


---
## Scenario 1: Create Base Table (Schema v0)

Start with a simple schema

In [7]:
# Drop table if exists (for re-runnability)
spark.sql(f"DROP TABLE IF EXISTS {FULL_TABLE}")

# Create the base Iceberg table
spark.sql(f"""
    CREATE TABLE {FULL_TABLE} (
        id BIGINT,
        name STRING,
        created_at TIMESTAMP
    )
    USING iceberg
""")

print(f"Created table: {FULL_TABLE}")
spark.sql(f"DESCRIBE {FULL_TABLE}").show(truncate=False)

Created table: glue_catalog.jpmc_demo_warehouse.events
+----------+---------+-------+
|col_name  |data_type|comment|
+----------+---------+-------+
|id        |bigint   |NULL   |
|name      |string   |NULL   |
|created_at|timestamp|NULL   |
+----------+---------+-------+


In [9]:
# Insert initial data (schema v0)
spark.sql(f"""
    INSERT INTO {FULL_TABLE} VALUES
        (1, 'alice', timestamp '2024-01-01 10:00:00'),
        (2, 'bob', timestamp '2024-01-02 11:00:00'),
        (3, 'charlie', timestamp '2024-01-03 12:00:00')
""")

print("Inserted 3 rows at schema v0")
spark.sql(f"SELECT * FROM {FULL_TABLE}").show()

Inserted 3 rows at schema v0
+---+-------+-------------------+
| id|   name|         created_at|
+---+-------+-------------------+
|  1|  alice|2024-01-01 10:00:00|
|  2|    bob|2024-01-02 11:00:00|
|  3|charlie|2024-01-03 12:00:00|
+---+-------+-------------------+


---
## Scenario 2: Add New Columns (Schema v0 -> v1)

Add  and  columns. Existing data will return  for these new columns.

In [11]:
# Add columns
spark.sql(f"""
    ALTER TABLE {FULL_TABLE} ADD COLUMNS (
        region STRING,
        event_type STRING
    )
""")

print("Schema after adding columns:")
spark.sql(f"DESCRIBE {FULL_TABLE}").show(truncate=False)

Schema after adding columns:
+----------+---------+-------+
|col_name  |data_type|comment|
+----------+---------+-------+
|id        |bigint   |NULL   |
|name      |string   |NULL   |
|created_at|timestamp|NULL   |
|region    |string   |NULL   |
|event_type|string   |NULL   |
+----------+---------+-------+


In [13]:
# Verify: old rows show null for new columns
print("Old rows now show null for new columns:")
spark.sql(f"SELECT * FROM {FULL_TABLE}").show()

Old rows now show null for new columns:
+---+-------+-------------------+------+----------+
| id|   name|         created_at|region|event_type|
+---+-------+-------------------+------+----------+
|  1|  alice|2024-01-01 10:00:00|  NULL|      NULL|
|  2|    bob|2024-01-02 11:00:00|  NULL|      NULL|
|  3|charlie|2024-01-03 12:00:00|  NULL|      NULL|
+---+-------+-------------------+------+----------+


In [15]:
# Insert new data with all columns (schema v1)
spark.sql(f"""
    INSERT INTO {FULL_TABLE} VALUES
        (4, 'diana', timestamp '2024-02-01 09:00:00', 'us-east-1', 'login'),
        (5, 'eve', timestamp '2024-02-02 10:00:00', 'eu-west-1', 'purchase')
""")

print("Inserted 2 rows at schema v1 (with region and event_type)")
spark.sql(f"SELECT * FROM {FULL_TABLE} ORDER BY id").show()

Inserted 2 rows at schema v1 (with region and event_type)
+---+-------+-------------------+---------+----------+
| id|   name|         created_at|   region|event_type|
+---+-------+-------------------+---------+----------+
|  1|  alice|2024-01-01 10:00:00|     NULL|      NULL|
|  2|    bob|2024-01-02 11:00:00|     NULL|      NULL|
|  3|charlie|2024-01-03 12:00:00|     NULL|      NULL|
|  4|  diana|2024-02-01 09:00:00|us-east-1|     login|
|  5|    eve|2024-02-02 10:00:00|eu-west-1|  purchase|
+---+-------+-------------------+---------+----------+


In [21]:
spark.sql(f"""
SELECT * from {FULL_TABLE}.snapshots """).show(truncate=False)

+-----------------------+-------------------+-------------------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                                                    |summary                                                           

---
## Scenario 3: Drop a Column (Schema v1 -> v2)

Drop . The column is hidden from reads but data files are not rewritten.

In [23]:
# Drop event_type column
spark.sql(f"ALTER TABLE {FULL_TABLE} DROP COLUMN event_type")

print("Schema after dropping event_type:")
spark.sql(f"DESCRIBE {FULL_TABLE}").show(truncate=False)

Schema after dropping event_type:
+----------+---------+-------+
|col_name  |data_type|comment|
+----------+---------+-------+
|id        |bigint   |NULL   |
|name      |string   |NULL   |
|created_at|timestamp|NULL   |
|region    |string   |NULL   |
+----------+---------+-------+


In [25]:
# Verify: event_type no longer appears in reads
print("event_type is gone from query results (data files still have it physically):")
spark.sql(f"SELECT * FROM {FULL_TABLE} ORDER BY id").show()

event_type is gone from query results (data files still have it physically):
+---+-------+-------------------+---------+
| id|   name|         created_at|   region|
+---+-------+-------------------+---------+
|  1|  alice|2024-01-01 10:00:00|     NULL|
|  2|    bob|2024-01-02 11:00:00|     NULL|
|  3|charlie|2024-01-03 12:00:00|     NULL|
|  4|  diana|2024-02-01 09:00:00|us-east-1|
|  5|    eve|2024-02-02 10:00:00|eu-west-1|
+---+-------+-------------------+---------+


---
## Scenario 4: Insert Old-Schema Data into the Evolved Table

Simulate loading data that only has the original v0 columns

### Approach A: Let Iceberg fill nulls for missing columns

In [27]:
# Simulate old data as a DataFrame (only v0 columns)
old_data_v0 = spark.createDataFrame([
    (100, "legacy_user_1", "2023-06-15 08:00:00"),
    (101, "legacy_user_2", "2023-06-16 09:00:00"),
    (102, "legacy_user_3", "2023-06-17 10:00:00"),
], ["id", "name", "created_at"])

old_data_v0 = old_data_v0.withColumn("id", col("id").cast(LongType()))     .withColumn("created_at", col("created_at").cast(TimestampType()))

# Add missing columns as null to match current schema
current_columns = [f.name for f in spark.table(FULL_TABLE).schema.fields]
print(f"Current table columns: {current_columns}")

for col_name in current_columns:
    if col_name not in old_data_v0.columns:
        old_data_v0 = old_data_v0.withColumn(col_name, lit(None).cast(StringType()))

# Write — region will be null
old_data_v0.select(current_columns).writeTo(FULL_TABLE).append()

print("Inserted old v0 data with nulls for missing columns:")
spark.sql(f"SELECT * FROM {FULL_TABLE} WHERE id >= 100 ORDER BY id").show()

Current table columns: ['id', 'name', 'created_at', 'region']
Inserted old v0 data with nulls for missing columns:
+---+-------------+-------------------+------+
| id|         name|         created_at|region|
+---+-------------+-------------------+------+
|100|legacy_user_1|2023-06-15 08:00:00|  NULL|
|101|legacy_user_2|2023-06-16 09:00:00|  NULL|
|102|legacy_user_3|2023-06-17 10:00:00|  NULL|
+---+-------------+-------------------+------+


### Approach B: Provide default values for missing columns

In [ ]:
# Simulate another batch of old data
old_data_v0_batch2 = spark.createDataFrame([
    (200, "backfill_user_1", "2023-03-01 07:00:00"),
    (201, "backfill_user_2", "2023-03-02 08:00:00"),
], ["id", "name", "created_at"])

old_data_v0_batch2 = old_data_v0_batch2     .withColumn("id", col("id").cast(LongType()))     .withColumn("created_at", col("created_at").cast(TimestampType()))     .withColumn("region", lit("UNKNOWN"))

old_data_v0_batch2.writeTo(FULL_TABLE).append()

print("Inserted old data with default region='UNKNOWN':")
spark.sql(f"SELECT * FROM {FULL_TABLE} WHERE id >= 200 ORDER BY id").show()

### Approach C: MERGE for upsert semantics

In [ ]:
# Create a temp view simulating old data that may overlap
overlap_data = spark.createDataFrame([
    (1, "alice_updated", "2024-01-01 10:00:00"),   # existing id=1
    (300, "new_user", "2023-12-01 06:00:00"),       # new record
], ["id", "name", "created_at"])

overlap_data = overlap_data     .withColumn("id", col("id").cast(LongType()))     .withColumn("created_at", col("created_at").cast(TimestampType()))

overlap_data.createOrReplaceTempView("old_events_staging")

spark.sql(f"""
    MERGE INTO {FULL_TABLE} AS target
    USING old_events_staging AS source
    ON target.id = source.id
    WHEN MATCHED THEN UPDATE SET
        name = source.name,
        created_at = source.created_at
    WHEN NOT MATCHED THEN INSERT (id, name, created_at, region)
        VALUES (source.id, source.name, source.created_at, NULL)
""")

print("After MERGE (id=1 updated, id=300 inserted):")
spark.sql(f"SELECT * FROM {FULL_TABLE} WHERE id IN (1, 300) ORDER BY id").show()

---
## Scenario 5: Combined Evolution — Add and Remove in Sequence

Add a  column (schema v2 -> v3), then insert v0-era data into v3.

In [29]:
# Add priority column
spark.sql(f"ALTER TABLE {FULL_TABLE} ADD COLUMNS (priority INT)")

print("Schema v3 (added priority):")
spark.sql(f"DESCRIBE {FULL_TABLE}").show(truncate=False)

Schema v3 (added priority):
+----------+---------+-------+
|col_name  |data_type|comment|
+----------+---------+-------+
|id        |bigint   |NULL   |
|name      |string   |NULL   |
|created_at|timestamp|NULL   |
|region    |string   |NULL   |
|priority  |int      |NULL   |
+----------+---------+-------+


In [31]:
# Insert data with all v3 columns
spark.sql(f"""
    INSERT INTO {FULL_TABLE} VALUES
        (6, 'frank', timestamp '2024-03-01 14:00:00', 'ap-southeast-1', 1),
        (7, 'grace', timestamp '2024-03-02 15:00:00', 'us-west-2', 3)
""")

print("Inserted rows with priority:")
spark.sql(f"SELECT * FROM {FULL_TABLE} ORDER BY id").show()

Inserted rows with priority:
+---+-------------+-------------------+--------------+--------+
| id|         name|         created_at|        region|priority|
+---+-------------+-------------------+--------------+--------+
|  1|        alice|2024-01-01 10:00:00|          NULL|    NULL|
|  2|          bob|2024-01-02 11:00:00|          NULL|    NULL|
|  3|      charlie|2024-01-03 12:00:00|          NULL|    NULL|
|  4|        diana|2024-02-01 09:00:00|     us-east-1|    NULL|
|  5|          eve|2024-02-02 10:00:00|     eu-west-1|    NULL|
|  6|        frank|2024-03-01 14:00:00|ap-southeast-1|       1|
|  7|        grace|2024-03-02 15:00:00|     us-west-2|       3|
|100|legacy_user_1|2023-06-15 08:00:00|          NULL|    NULL|
|101|legacy_user_2|2023-06-16 09:00:00|          NULL|    NULL|
|102|legacy_user_3|2023-06-17 10:00:00|          NULL|    NULL|
+---+-------------+-------------------+--------------+--------+


In [33]:
# Insert v0-era data into v3 table — align schema
old_v0_for_v3 = spark.createDataFrame([
    (400, "archive_user_1", "2022-11-01 05:00:00"),
    (401, "archive_user_2", "2022-11-02 06:00:00"),
], ["id", "name", "created_at"])

aligned_df = old_v0_for_v3     .withColumn("id", col("id").cast(LongType()))     .withColumn("created_at", col("created_at").cast(TimestampType()))     .withColumn("region", lit(None).cast(StringType()))     .withColumn("priority", lit(None).cast(IntegerType()))

aligned_df.writeTo(FULL_TABLE).append()

print("Inserted v0-era data into v3 table (region=null, priority=null):")
spark.sql(f"SELECT * FROM {FULL_TABLE} WHERE id >= 400 ORDER BY id").show()

Inserted v0-era data into v3 table (region=null, priority=null):
+---+--------------+-------------------+------+--------+
| id|          name|         created_at|region|priority|
+---+--------------+-------------------+------+--------+
|400|archive_user_1|2022-11-01 05:00:00|  NULL|    NULL|
|401|archive_user_2|2022-11-02 06:00:00|  NULL|    NULL|
+---+--------------+-------------------+------+--------+


In [35]:
spark.sql(f"SELECT * FROM {FULL_TABLE} ORDER BY id").show(truncate=False)

+---+--------------+-------------------+--------------+--------+
|id |name          |created_at         |region        |priority|
+---+--------------+-------------------+--------------+--------+
|1  |alice         |2024-01-01 10:00:00|NULL          |NULL    |
|2  |bob           |2024-01-02 11:00:00|NULL          |NULL    |
|3  |charlie       |2024-01-03 12:00:00|NULL          |NULL    |
|4  |diana         |2024-02-01 09:00:00|us-east-1     |NULL    |
|5  |eve           |2024-02-02 10:00:00|eu-west-1     |NULL    |
|6  |frank         |2024-03-01 14:00:00|ap-southeast-1|1       |
|7  |grace         |2024-03-02 15:00:00|us-west-2     |3       |
|100|legacy_user_1 |2023-06-15 08:00:00|NULL          |NULL    |
|101|legacy_user_2 |2023-06-16 09:00:00|NULL          |NULL    |
|102|legacy_user_3 |2023-06-17 10:00:00|NULL          |NULL    |
|400|archive_user_1|2022-11-01 05:00:00|NULL          |NULL    |
|401|archive_user_2|2022-11-02 06:00:00|NULL          |NULL    |
+---+--------------+-----

---
## Scenario 6: Using Branching for Safe Schema Evolution

Create a branch, write data there, validate, then fast-forward main.

> **Note:** Branching requires Iceberg 1.2+ and Spark 3.4+. AWS Glue 5.0 supports this.

In [37]:
# Create a branch from the current state
spark.sql(f"""
    ALTER TABLE {FULL_TABLE}
    CREATE BRANCH schema_test

""")

print("Created branch 'schema_test'")

Created branch 'schema_test'


In [45]:
#spark.sql(f"SELECT * FROM {FULL_TABLE}.branch_schema_test ORDER BY id").show(truncate=False)
spark.sql(f"SELECT * FROM {FULL_TABLE}.refs  ").show(truncate=False)

+-----------+------+-------------------+-----------------------+---------------------+----------------------+
|name       |type  |snapshot_id        |max_reference_age_in_ms|min_snapshots_to_keep|max_snapshot_age_in_ms|
+-----------+------+-------------------+-----------------------+---------------------+----------------------+
|schema_test|BRANCH|8506684126949114473|NULL                   |NULL                 |NULL                  |
|main       |BRANCH|8506684126949114473|NULL                   |NULL                 |NULL                  |
+-----------+------+-------------------+-----------------------+---------------------+----------------------+


In [47]:
# Insert more data with all v3 columns to MAIN
spark.sql(f"""
    INSERT INTO {FULL_TABLE} VALUES
        (8, 'peter', timestamp '2025-03-01 14:00:00', 'us-east-1', 2),
        (9, 'parker', timestamp '2025-03-02 15:00:00', 'us-west-2', 4)
""")

print("Inserted rows with priority:")
spark.sql(f"SELECT * FROM {FULL_TABLE} ORDER BY id").show()

Inserted rows with priority:
+---+--------------+-------------------+--------------+--------+
| id|          name|         created_at|        region|priority|
+---+--------------+-------------------+--------------+--------+
|  1|         alice|2024-01-01 10:00:00|          NULL|    NULL|
|  2|           bob|2024-01-02 11:00:00|          NULL|    NULL|
|  3|       charlie|2024-01-03 12:00:00|          NULL|    NULL|
|  4|         diana|2024-02-01 09:00:00|     us-east-1|    NULL|
|  5|           eve|2024-02-02 10:00:00|     eu-west-1|    NULL|
|  6|         frank|2024-03-01 14:00:00|ap-southeast-1|       1|
|  7|         grace|2024-03-02 15:00:00|     us-west-2|       3|
|  8|         peter|2025-03-01 14:00:00|     us-east-1|       2|
|  9|        parker|2025-03-02 15:00:00|     us-west-2|       4|
|100| legacy_user_1|2023-06-15 08:00:00|          NULL|    NULL|
|101| legacy_user_2|2023-06-16 09:00:00|          NULL|    NULL|
|102| legacy_user_3|2023-06-17 10:00:00|          NULL|    NU

In [51]:
spark.sql(f"SELECT * FROM {FULL_TABLE}.branch_main ORDER BY id").show(truncate=False)


+---+--------------+-------------------+--------------+--------+
|id |name          |created_at         |region        |priority|
+---+--------------+-------------------+--------------+--------+
|1  |alice         |2024-01-01 10:00:00|NULL          |NULL    |
|2  |bob           |2024-01-02 11:00:00|NULL          |NULL    |
|3  |charlie       |2024-01-03 12:00:00|NULL          |NULL    |
|4  |diana         |2024-02-01 09:00:00|us-east-1     |NULL    |
|5  |eve           |2024-02-02 10:00:00|eu-west-1     |NULL    |
|6  |frank         |2024-03-01 14:00:00|ap-southeast-1|1       |
|7  |grace         |2024-03-02 15:00:00|us-west-2     |3       |
|8  |peter         |2025-03-01 14:00:00|us-east-1     |2       |
|9  |parker        |2025-03-02 15:00:00|us-west-2     |4       |
|100|legacy_user_1 |2023-06-15 08:00:00|NULL          |NULL    |
|101|legacy_user_2 |2023-06-16 09:00:00|NULL          |NULL    |
|102|legacy_user_3 |2023-06-17 10:00:00|NULL          |NULL    |
|400|archive_user_1|2022-

In [ ]:
# Insert OLD data with all columns (schema v1)
spark.sql(f"""
    INSERT INTO {FULL_TABLE} VALUES
        (4, 'diana', timestamp '2024-02-01 09:00:00', 'us-east-1', 'login'),
        (5, 'eve', timestamp '2024-02-02 10:00:00', 'eu-west-1', 'purchase')
""")

print("Inserted 2 rows at schema v1 (with region and event_type)")
spark.sql(f"SELECT * FROM {FULL_TABLE} ORDER BY id").show()

In [ ]:
# Write OLD data to the branch only
spark.conf.set("spark.wap.branch", "schema_test")

branch_data = spark.createDataFrame([
    (500, "branch_user_1", "2024-04-01 10:00:00", "us-east-1", 2),
    (501, "branch_user_2", "2024-04-02 11:00:00", "eu-west-1", 1),
], ["id", "name", "created_at", "region", "priority"])

branch_data = branch_data     .withColumn("id", col("id").cast(LongType()))     .withColumn("created_at", col("created_at").cast(TimestampType()))     .withColumn("priority", col("priority").cast(IntegerType()))

branch_data.writeTo(FULL_TABLE).append()

# Reset WAP branch
spark.conf.unset("spark.wap.branch")

print("Wrote 2 rows to branch 'schema_test'")

In [ ]:
# Validate: branch has the new rows, main does not
print("=== Reading from branch 'schema_test' ===")
spark.read.option("branch", "schema_test").table(FULL_TABLE)     .filter("id >= 500").show()

print("=== Reading from main (should NOT have ids 500, 501) ===")
spark.sql(f"SELECT * FROM {FULL_TABLE} WHERE id >= 500").show()

In [ ]:
# Fast-forward main to include the branch data
spark.sql(f"""
    CALL glue_catalog.system.fast_forward(
        table => '{DATABASE}.{TABLE_NAME}',
        branch => 'main',
        to => 'schema_test'
    )
""")

print("Fast-forwarded main to schema_test")
print("Main now has the branch data:")
spark.sql(f"SELECT * FROM {FULL_TABLE} WHERE id >= 500 ORDER BY id").show()

In [ ]:
# Clean up the branch
spark.sql(f"ALTER TABLE {FULL_TABLE} DROP BRANCH schema_test")
print("Dropped branch 'schema_test'")

---
## Scenario 7: Loading Old Data That Contains Dropped Columns

The table has dropped  (back in Scenario 3). Now we need to load data that still has that column.

Iceberg rejects writes for columns not in the current schema, so we must handle the mismatch.

### Approach A: Drop extra columns from the data before insert (lossy)

In [ ]:
# Simulate old data that has the dropped column event_type
old_data_with_dropped_cols = spark.createDataFrame([
    (600, "old_full_1", "2023-01-10 08:00:00", "us-west-2", "click", 5),
    (601, "old_full_2", "2023-01-11 09:00:00", "eu-west-1", "view", 2),
], ["id", "name", "created_at", "region", "event_type", "priority"])

old_data_with_dropped_cols = old_data_with_dropped_cols     .withColumn("id", col("id").cast(LongType()))     .withColumn("created_at", col("created_at").cast(TimestampType()))     .withColumn("priority", col("priority").cast(IntegerType()))

# Get current table columns and select only those
current_columns = [f.name for f in spark.table(FULL_TABLE).schema.fields]
print(f"Current table columns: {current_columns}")

# Drop columns not in current schema
trimmed_df = old_data_with_dropped_cols.select(current_columns)
print("Trimmed DataFrame (event_type removed):")
trimmed_df.show()

trimmed_df.writeTo(FULL_TABLE).append()
print("Inserted (event_type discarded):")
spark.sql(f"SELECT * FROM {FULL_TABLE} WHERE id >= 600 ORDER BY id").show()

### Approach B: Re-add dropped columns temporarily, load, then drop again (lossless)

In [ ]:
# Re-add the dropped column temporarily
spark.sql(f"ALTER TABLE {FULL_TABLE} ADD COLUMNS (event_type STRING)")

print("Schema after re-adding event_type:")
spark.sql(f"DESCRIBE {FULL_TABLE}").show(truncate=False)

In [ ]:
# Now we can insert data that includes event_type
old_data_full_schema = spark.createDataFrame([
    (700, "archive_full_1", "2022-08-01 07:00:00", "ap-south-1", 3, "signup"),
    (701, "archive_full_2", "2022-08-02 08:00:00", "us-east-1", 1, "logout"),
], ["id", "name", "created_at", "region", "priority", "event_type"])

old_data_full_schema = old_data_full_schema     .withColumn("id", col("id").cast(LongType()))     .withColumn("created_at", col("created_at").cast(TimestampType()))     .withColumn("priority", col("priority").cast(IntegerType()))

old_data_full_schema.writeTo(FULL_TABLE).append()

print("Inserted with event_type preserved:")
spark.sql(f"SELECT * FROM {FULL_TABLE} WHERE id >= 700 ORDER BY id").show()

In [ ]:
# Drop event_type again to restore the production schema
spark.sql(f"ALTER TABLE {FULL_TABLE} DROP COLUMN event_type")

print("Schema restored (event_type dropped again):")
spark.sql(f"DESCRIBE {FULL_TABLE}").show(truncate=False)

print("Data still readable (event_type hidden):")
spark.sql(f"SELECT * FROM {FULL_TABLE} WHERE id >= 700 ORDER BY id").show()

### Approach C: Use a staging table

In [ ]:
STAGING_TABLE = f"glue_catalog.{DATABASE}.events_staging"

# Create staging table with the old full schema
spark.sql(f"DROP TABLE IF EXISTS {STAGING_TABLE}")
spark.sql(f"""
    CREATE TABLE {STAGING_TABLE} (
        id BIGINT,
        name STRING,
        created_at TIMESTAMP,
        region STRING,
        event_type STRING,
        priority INT
    )
    USING iceberg
""")

print(f"Created staging table: {STAGING_TABLE}")

In [ ]:
# Load old data into staging (preserves all columns including dropped ones)
staging_data = spark.createDataFrame([
    (800, "staged_1", "2022-05-01 06:00:00", "us-west-1", "download", 4),
    (801, "staged_2", "2022-05-02 07:00:00", "eu-central-1", "upload", 2),
], ["id", "name", "created_at", "region", "event_type", "priority"])

staging_data = staging_data     .withColumn("id", col("id").cast(LongType()))     .withColumn("created_at", col("created_at").cast(TimestampType()))     .withColumn("priority", col("priority").cast(IntegerType()))

staging_data.writeTo(STAGING_TABLE).append()

print("Staging table has full old-schema data:")
spark.sql(f"SELECT * FROM {STAGING_TABLE}").show()

In [ ]:
# Insert into production selecting only current columns
current_columns_str = ", ".join([f.name for f in spark.table(FULL_TABLE).schema.fields])
print(f"Inserting columns: {current_columns_str}")

spark.sql(f"""
    INSERT INTO {FULL_TABLE} ({current_columns_str})
    SELECT {current_columns_str} FROM {STAGING_TABLE}
""")

print("Production table after staging insert:")
spark.sql(f"SELECT * FROM {FULL_TABLE} WHERE id >= 800 ORDER BY id").show()

### Approach D: Branch + Temporary Schema Expansion

Combine branching with temporary column re-addition for zero-downtime backfill.

In [ ]:
# 1. Create a branch for the backfill
spark.sql(f"""
    ALTER TABLE {FULL_TABLE}
    CREATE BRANCH backfill_old_data
    RETAIN 7 DAYS
""")

# 2. Re-add dropped column
spark.sql(f"ALTER TABLE {FULL_TABLE} ADD COLUMNS (event_type STRING)")

print("Branch created, event_type re-added for backfill")
spark.sql(f"DESCRIBE {FULL_TABLE}").show(truncate=False)

In [ ]:
# 3. Write old data to the branch
spark.conf.set("spark.wap.branch", "backfill_old_data")

backfill_data = spark.createDataFrame([
    (900, "branched_backfill_1", "2022-01-15 05:00:00", "us-east-2", 5, "api_call"),
    (901, "branched_backfill_2", "2022-01-16 06:00:00", "ap-northeast-1", 3, "webhook"),
], ["id", "name", "created_at", "region", "priority", "event_type"])

backfill_data = backfill_data     .withColumn("id", col("id").cast(LongType()))     .withColumn("created_at", col("created_at").cast(TimestampType()))     .withColumn("priority", col("priority").cast(IntegerType()))

backfill_data.writeTo(FULL_TABLE).append()
spark.conf.unset("spark.wap.branch")

print("Wrote backfill data to branch")

In [ ]:
# 4. Drop event_type again (hides from future reads)
spark.sql(f"ALTER TABLE {FULL_TABLE} DROP COLUMN event_type")

# 5. Validate on the branch
print("Branch data (event_type hidden after drop):")
spark.read.option("branch", "backfill_old_data").table(FULL_TABLE)     .filter("id >= 900").show()

print("Main still unchanged:")
spark.sql(f"SELECT * FROM {FULL_TABLE} WHERE id >= 900").show()

In [ ]:
# 6. Fast-forward main
spark.sql(f"""
    CALL glue_catalog.system.fast_forward(
        table => '{DATABASE}.{TABLE_NAME}',
        branch => 'main',
        to => 'backfill_old_data'
    )
""")

print("Fast-forwarded main. Backfill data now visible:")
spark.sql(f"SELECT * FROM {FULL_TABLE} WHERE id >= 900 ORDER BY id").show()

In [ ]:
# 7. Clean up branch
spark.sql(f"ALTER TABLE {FULL_TABLE} DROP BRANCH backfill_old_data")
print("Dropped branch 'backfill_old_data'")

---
## Final State & Summary

In [ ]:
# Show final table state
print("=== Final Schema ===")
spark.sql(f"DESCRIBE {FULL_TABLE}").show(truncate=False)

print(f"=== Total Row Count ===")
spark.sql(f"SELECT count(*) as total_rows FROM {FULL_TABLE}").show()

print("=== Sample of All Data ===")
spark.sql(f"SELECT * FROM {FULL_TABLE} ORDER BY id").show(50, truncate=False)

In [ ]:
# View schema evolution history via snapshots
print("=== Snapshot History ===")
spark.sql(f"SELECT * FROM {FULL_TABLE}.snapshots").show(truncate=False)

---
## Cleanup (Optional)

Uncomment and run to drop the demo tables.

In [ ]:
# spark.sql(f"DROP TABLE IF EXISTS {FULL_TABLE}")
# spark.sql(f"DROP TABLE IF EXISTS {STAGING_TABLE}")
# spark.sql(f"DROP DATABASE IF EXISTS glue_catalog.{DATABASE}")
# print("Cleaned up all demo resources")